# Частина 2: Аналіз споживання електроенергії

### 1. Завантаження та підготовка датасету
**Завдання:** Звантажити та відкрити датасет Individual Household Electric Power Consumption. Здійснити data cleaning (обробка пропущених значень).

In [1]:
import urllib.request
import zipfile
import os
import pandas as pd

dataset_dir = "power_data"
os.makedirs(dataset_dir, exist_ok=True)

zip_path = os.path.join(dataset_dir, "power.zip")
csv_path = os.path.join(dataset_dir, "household_power_consumption.txt")

if not os.path.exists(csv_path):
    print("Завантажуємо архів...")
    url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
    urllib.request.urlretrieve(url, zip_path)
    
    print("Розпаковуємо...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dataset_dir)
    print("Готово!")
else:
    print("Датасет вже завантажено!")

Завантажуємо архів...
Розпаковуємо...
Готово!


### 2. Очистка даних (Data Cleaning)
**Завдання:** Здійснити data cleaning (обробка пропущених значень).

In [2]:
def load_power(filepath):
    df = pd.read_csv(filepath, sep=';', na_values=['?'], low_memory=False)
    df = df.dropna().reset_index(drop=True)
    print(f"Рядків після очистки: {len(df)}")
    return df

power_df = load_power(csv_path)
power_df.head()

Рядків після очистки: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### 3.1. Вибірка: Потужність більше 5 кВт
**Завдання:** Обрати всі домогосподарства, у яких загальна активна споживана потужність перевищує 5 кВт.

In [3]:
power_df['Global_active_power'] = pd.to_numeric(power_df['Global_active_power'])

high_power = power_df[power_df['Global_active_power'] > 5.0]
print(f"знайшла {len(high_power)} записів")
high_power.head()

знайшла 17547 записів


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### 3.2. Вибірка: Напруга більше 235 В
**Завдання:** Обрати всі домогосподарства, у яких напруга перевищує 235 В.

In [4]:
power_df['Voltage'] = pd.to_numeric(power_df['Voltage'])
high_voltage = power_df[power_df['Voltage'] > 235.0]

print(f"знайшла {len(high_voltage)} записів")
high_voltage.head()

знайшла 1952491 записів


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
5,16/12/2006,17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0
6,16/12/2006,17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0
7,16/12/2006,17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0
14,16/12/2006,17:38:00,4.054,0.422,235.24,17.6,0.0,1.0,17.0


### 3.3. Складна вибірка за кількома умовами
**Завдання:** Обрати всі домогосподарства, у яких сила струму в межах 19-21 А, а споживання sub_metering_2 перевищує sub_metering_3.

In [5]:
power_df['Global_intensity'] = pd.to_numeric(power_df['Global_intensity'])

filtered = power_df[
    (power_df['Global_intensity'] >= 19) & 
    (power_df['Global_intensity'] <= 21) & 
    (power_df['Sub_metering_2'] > power_df['Sub_metering_3'])
]
print(f"знайшла {len(filtered)} записів")
filtered.head()

знайшла 4289 записів


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
40,16/12/2006,18:04:00,4.928,0.202,235.01,21.0,0.0,37.0,16.0
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
466,17/12/2006,01:10:00,4.860,0.198,239.65,20.2,0.0,36.0,0.0


### 4. Випадкова вибірка 500,000 записів
**Завдання:** З початкового датасету (після очистки) обрати випадковим чином 500,000 домогосподарств.

In [6]:
sample_df = power_df.sample(n=500000).reset_index(drop=True)
print(f"вибірка: {len(sample_df)} записів")
sample_df.head()

вибірка: 500000 записів


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,15/12/2009,06:17:00,0.288,0.000,246.69,1.2,0.0,0.0,0.0
1,5/6/2009,14:58:00,2.172,0.654,234.84,10.0,0.0,0.0,18.0
2,9/11/2009,07:01:00,2.814,0.286,241.67,11.6,0.0,2.0,19.0
3,8/7/2008,14:51:00,0.184,0.000,240.14,1.0,0.0,0.0,1.0
4,17/6/2007,04:32:00,0.174,0.078,244.01,0.8,0.0,0.0,0.0


### 5. Середнє споживання по групах (sub_metering)
**Завдання:** З отриманої вибірки (500,000 записів) порахувати середні значення для трьох груп споживання (Sub_metering_1, Sub_metering_2, Sub_metering_3).

In [7]:
means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
print("середні значення:")
print(means)

середні значення:
Sub_metering_1    1.107880
Sub_metering_2    1.301146
Sub_metering_3    6.456028
dtype: float64
